# 多模态人脸情感识别训练脚手架（VR / Meta Quest Pro）

图像分支（EfficientNet-B2）+ 表情激活分支（63 维 FEA → MLP）后期融合，分类到 **7 类情感**。
对齐 **EmoHeVRDB**（Meta Quest Pro 采集，与本设备同源）的数据格式，也可直接喂入你自采的 Quest Pro 数据。

> **数据未就绪也能跑**：把下方 `USE_DUMMY_DATA = True`，整本 notebook 会用随机数据端到端跑通，
> 验证模型/训练/评估管线无误。拿到 EmoHeVRDB（或自采数据）后改为 `False` 并实现 `build_manifest`。

**获取 EmoHeVRDB**：见同目录 `VR_datasets_access_guide.md`（需学术 EULA 申请）。

**依赖**（已在 conda 环境 `webFaceEmotionRec` 中）：`torch torchvision timm numpy pillow tqdm`。


## 1. 配置

In [ ]:
import os

# ── 数据 ──────────────────────────────────────────────────────────────
USE_DUMMY_DATA = True          # True: 随机数据冒烟测试；False: 读取真实 EmoHeVRDB / 自采数据
EMOHEVRDB_ROOT = r"d:\Project\webFaceEmotionRec\DataSet\EmoHeVRDB"  # 解压后的根目录（拿到数据后填）
CKPT_DIR       = r"d:\Project\webFaceEmotionRec\DataSet\_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# ── 7 类（字母序，与 DataSet/fer2013 的 ImageFolder 一致）────────────────
CLASS_NAMES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

# EmoHeVRDB 原始情感名 → 本项目 7 类 key
EMOHEVR_LABEL_MAP = {
    "anger": "angry", "disgust": "disgust", "fear": "fear", "happiness": "happy",
    "neutral": "neutral", "sadness": "sad", "surprise": "surprise",
}

# ── 模态维度 ─────────────────────────────────────────────────────────
IMG_SIZE = 224                 # EmoHeVRDB-SI 为 224x224
FEA_DIM  = 63                  # Quest Pro Face Tracking API 的 FACS blendshape 维度

# ── 训练超参 ─────────────────────────────────────────────────────────
BATCH_SIZE   = 32
EPOCHS       = 15
LR           = 1e-4
WEIGHT_DECAY = 1e-4
FREEZE_BACKBONE_EPOCHS = 2     # 前若干 epoch 冻结图像主干，只训融合头与 FEA 分支
NUM_WORKERS  = 0               # Windows 上建议 0；Linux 可调大
SEED         = 42


In [ ]:
import random
import numpy as np
import torch

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("设备:", DEVICE, "| torch", torch.__version__)


## 2. 数据集

每个样本 = `(图像[3,224,224], FEA[63], 标签)`。

- **真实数据**：实现 `build_manifest()`，返回 `[{"image": 路径, "fea": json路径, "label": 类名}, ...]`。
  EmoHeVRDB 的具体目录/命名规范见其 GitHub 仓库文档；SFEA 是每个样本一个 63 维 JSON 向量，
  SI 是 224×224 jpg。**务必保证 FEA 的 63 维顺序与 EmoHeVRDB 一致**（自采数据按同样的
  `OVRFaceExpressions` 枚举顺序存），否则两数据集无法合并训练。
- **冒烟测试**：`DummyDataset` 直接产出随机张量，跳过磁盘 IO。


In [ ]:
import glob
import json
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def build_manifest(root, split):
    """读取自采数据（DataSet/quest_pro_capture_spec.md 格式）或 EmoHeVRDB。

    期望目录：<root>/<split>/<sample_id>.json + <sample_id>_central.jpg
    每个 JSON 至少含 {"label", "image_central", "blendshapes":[63]}。
    返回 [{"image": 图像绝对路径, "fea": json路径, "label": 原始情感名/key}, ...]。

    注：EmoHeVRDB 官方布局可能不同（SI 图像与 SFEA 向量分目录）——拿到数据后按其
    GitHub 命名规范微调本函数即可，Dataset 部分无需改。
    """
    split_dir = os.path.join(root, split)
    if not os.path.isdir(split_dir):
        raise FileNotFoundError(f"找不到 {split_dir}；请确认 EMOHEVRDB_ROOT 与 split 目录存在。")

    manifest = []
    for jp in sorted(glob.glob(os.path.join(split_dir, "*.json"))):
        with open(jp, "r", encoding="utf-8") as f:
            meta = json.load(f)
        img_name = meta.get("image_central") or meta.get("image")
        if not img_name:
            continue
        img_path = os.path.join(split_dir, img_name)
        if not os.path.exists(img_path):
            print(f"[warn] 缺图像，跳过: {img_path}")
            continue
        manifest.append({"image": img_path, "fea": jp, "label": meta["label"]})

    if not manifest:
        raise RuntimeError(f"{split_dir} 下未找到有效样本（*.json + 对应图像）。")
    print(f"[manifest] {split}: {len(manifest)} 样本")
    return manifest


def _load_fea(fea_path):
    """FEA JSON 可为 63 维裸 list（EmoHeVRDB SFEA），或含 'blendshapes' 字段的 dict（自采）。"""
    with open(fea_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    vec = data["blendshapes"] if isinstance(data, dict) else data
    return torch.tensor(vec, dtype=torch.float32)


class MultimodalFERDataset(Dataset):
    """真实数据集：图像 + 63 维 FEA + 标签。"""

    def __init__(self, manifest, transform):
        self.items = manifest
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        it = self.items[i]
        img = self.transform(Image.open(it["image"]).convert("RGB"))
        fea = _load_fea(it["fea"])                   # 63 维，范围 0~1
        assert fea.numel() == FEA_DIM, f"FEA 维度应为 {FEA_DIM}，实际 {fea.numel()}"
        label = CLASS_TO_IDX[EMOHEVR_LABEL_MAP.get(it["label"], it["label"])]
        return img, fea, label


class DummyDataset(Dataset):
    """冒烟测试：随机图像 + 随机 FEA + 随机标签，形状与真实一致。"""

    def __init__(self, n=256):
        g = torch.Generator().manual_seed(SEED)
        self.imgs = torch.randn(n, 3, IMG_SIZE, IMG_SIZE, generator=g)
        self.fea = torch.rand(n, FEA_DIM, generator=g)
        self.labels = torch.randint(0, NUM_CLASSES, (n,), generator=g)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.imgs[i], self.fea[i], int(self.labels[i])


In [ ]:
from torch.utils.data import DataLoader

if USE_DUMMY_DATA:
    train_ds, val_ds = DummyDataset(256), DummyDataset(64)
else:
    train_ds = MultimodalFERDataset(build_manifest(EMOHEVRDB_ROOT, "train"), train_tf)
    val_ds   = MultimodalFERDataset(build_manifest(EMOHEVRDB_ROOT, "val"),   eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS)
print(f"train={len(train_ds)}  val={len(val_ds)}")


In [ ]:
# 类别权重（FER 类别不均衡，如 disgust 远少于 happy）——按训练集频次反比加权
def compute_class_weights(dataset):
    counts = torch.zeros(NUM_CLASSES)
    for i in range(len(dataset)):
        counts[dataset[i][2]] += 1
    counts = counts.clamp(min=1)
    w = counts.sum() / (NUM_CLASSES * counts)
    return w

class_weights = compute_class_weights(train_ds).to(DEVICE)
print("类别权重:", {c: round(float(w), 3) for c, w in zip(CLASS_NAMES, class_weights)})


## 3. 模型：双分支后期融合

```
图像 ─► EfficientNet-B2 (timm, 预训练) ─► 1408-d ┐
                                                 ├─ concat ─► 融合头 ─► 7 类
FEA  ─► MLP(63→128)                    ─►  128-d ┘
```


In [ ]:
import timm
import torch.nn as nn


class MultimodalFER(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, fea_dim=FEA_DIM, pretrained=True):
        super().__init__()
        # 图像分支：EfficientNet-B2 主干（num_classes=0 → 输出池化后特征）
        self.image_backbone = timm.create_model(
            "efficientnet_b2", pretrained=pretrained, num_classes=0, global_pool="avg")
        img_feat = self.image_backbone.num_features        # 1408

        # FEA 分支
        self.fea_branch = nn.Sequential(
            nn.Linear(fea_dim, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 128), nn.ReLU(inplace=True),
        )

        # 融合头
        self.head = nn.Sequential(
            nn.Linear(img_feat + 128, 256), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def set_backbone_frozen(self, frozen: bool):
        for p in self.image_backbone.parameters():
            p.requires_grad = not frozen

    def forward(self, img, fea):
        fi = self.image_backbone(img)
        ff = self.fea_branch(fea)
        return self.head(torch.cat([fi, ff], dim=1))


model = MultimodalFER(pretrained=not USE_DUMMY_DATA).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"模型参数: {n_params:.1f}M")


## 4. 训练与评估

指标用 numpy/torch 手算（环境未装 sklearn）：准确率、macro-F1、混淆矩阵。


In [ ]:
import torch.nn.functional as F
from tqdm.auto import tqdm

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, all_pred, all_true = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for img, fea, label in tqdm(loader, leave=False):
            img, fea, label = img.to(DEVICE), fea.to(DEVICE), label.to(DEVICE)
            logits = model(img, fea)
            loss = criterion(logits, label)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * label.size(0)
            all_pred.append(logits.argmax(1).cpu())
            all_true.append(label.cpu())
    pred = torch.cat(all_pred); true = torch.cat(all_true)
    return total_loss / len(true), pred, true


def metrics(pred, true):
    acc = (pred == true).float().mean().item()
    cm = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)
    for t, p in zip(true, pred):
        cm[t, p] += 1
    f1s = []
    for c in range(NUM_CLASSES):
        tp = cm[c, c].item()
        fp = cm[:, c].sum().item() - tp
        fn = cm[c, :].sum().item() - tp
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec  = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    return acc, sum(f1s) / len(f1s), cm


In [ ]:
best_f1 = 0.0
for epoch in range(1, EPOCHS + 1):
    model.set_backbone_frozen(epoch <= FREEZE_BACKBONE_EPOCHS)

    tr_loss, tr_pred, tr_true = run_epoch(train_loader, train=True)
    va_loss, va_pred, va_true = run_epoch(val_loader, train=False)
    tr_acc, _, _ = metrics(tr_pred, tr_true)
    va_acc, va_f1, _ = metrics(va_pred, va_true)

    print(f"[{epoch:02d}/{EPOCHS}] train_loss={tr_loss:.3f} acc={tr_acc:.3f} | "
          f"val_loss={va_loss:.3f} acc={va_acc:.3f} macroF1={va_f1:.3f}")

    if va_f1 > best_f1:
        best_f1 = va_f1
        torch.save({"model": model.state_dict(), "classes": CLASS_NAMES,
                    "epoch": epoch, "val_f1": va_f1},
                   os.path.join(CKPT_DIR, "multimodal_fer_best.pt"))
        print(f"   [best] 保存最佳模型 (macroF1={va_f1:.3f})")


In [ ]:
# 验证集混淆矩阵
_, va_pred, va_true = run_epoch(val_loader, train=False)
va_acc, va_f1, cm = metrics(va_pred, va_true)
print(f"val acc={va_acc:.3f}  macroF1={va_f1:.3f}\n")
hdr = "真\\预 " + " ".join(f"{c[:4]:>5}" for c in CLASS_NAMES)
print(hdr)
for i, c in enumerate(CLASS_NAMES):
    print(f"{c[:4]:>5} " + " ".join(f"{cm[i, j].item():>5}" for j in range(NUM_CLASSES)))


## 5. 单样本推理示例

In [ ]:
@torch.no_grad()
def predict(img_tensor, fea_vector):
    """img_tensor: [3,224,224] 已归一化；fea_vector: 长度 63 的 list/tensor。"""
    model.eval()
    img = img_tensor.unsqueeze(0).to(DEVICE)
    fea = torch.as_tensor(fea_vector, dtype=torch.float32).view(1, -1).to(DEVICE)
    prob = F.softmax(model(img, fea), dim=1)[0].cpu()
    top = prob.argmax().item()
    return CLASS_NAMES[top], {c: round(float(p), 3) for c, p in zip(CLASS_NAMES, prob)}

# 用一条 dummy/val 样本演示
img0, fea0, lbl0 = val_ds[0]
pred_name, probs = predict(img0, fea0)
print("预测:", pred_name, "| 真值:", CLASS_NAMES[lbl0] if not USE_DUMMY_DATA else f"(dummy={CLASS_NAMES[lbl0]})")
print("概率:", probs)


## 6. 接入真实数据 / 自采 Quest Pro 数据的步骤

`build_manifest()` 已实现为读取 **`DataSet/quest_pro_capture_spec.md`** 定义的采集格式
（`<root>/<split>/<sample_id>.json` + `<sample_id>_central.jpg`），并兼容 EmoHeVRDB 的裸向量 JSON。

1. **自采（Meta Quest Pro）**：用 **`DataSet/QuestProFaceCapture.cs`**（Meta XR Movement SDK
   Face Tracking）录 63 维 blendshape 写 JSON；外接相机同步拍 224×224 下半脸图，按同一
   `sample_id` 命名。按受试者划分 train/val/test（避免身份泄漏）。
2. **EmoHeVRDB**：按 `VR_datasets_access_guide.md` 申请，解压后把 SI 图像 + SFEA 向量整理成
   上述布局（或按其官方目录微调 `build_manifest()`）。两数据集 **blendshape 顺序一致** 时可合并。
3. 设 `EMOHEVRDB_ROOT = <你的数据根>`、`USE_DUMMY_DATA = False`，从头重跑全 notebook。
4. **可选**：图像主干用项目 HSEmotion `enet_b2_7` 权重初始化，迁移学习更快收敛
   （HSEmotion 底层同为 timm enet_b2）。

> **GPU**：本机已装 CUDA 版 PyTorch（cu128，RTX 5090 sm_120），notebook 会自动用 GPU 训练。
